# TARA Week 9 — Docstring → tokens → tool call → tokens

1. the tool you write (`calculate`) becomes a JSON description,
2. that JSON becomes tokens in the prompt,
3. the model **only outputs text** — the tool call is a string until the harness parses it,
4. the harness: `parse_tool_calls` → `execute_tools` → append → generate again, in a loop,
5. a tool that calls Wikipedia: the request happens inside `execute_tools`,
6. the same loop through an API model (gpt-4o-mini),
7. the Wikipedia page object the WikiGame tools are built on.

Runtime: **GPU → T4** (Runtime ▸ Change runtime type).

## 0. Setup

In [ ]:
%pip -q install tiktoken transformers jinja2 openai accelerate wikipedia

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os, json, re, inspect, ast

# Only needed for section 7 (gpt-4o-mini via OpenRouter).
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("OPENROUTER_API_KEY loaded")
except Exception as e:
    print("No OPENROUTER_API_KEY in Colab secrets — section 7 will be skipped.")

OPENROUTER_API_KEY loaded


## 1. The tool

Same docstring as the slide.

In [ ]:
def evaluate_expression(expression: str) -> float:
    """Safely evaluate + - * / ** and parentheses. Anything else raises."""
    allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
               ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd, ast.Mod, ast.FloorDiv)
    tree = ast.parse(expression, mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, allowed):
            raise ValueError(f"not allowed: {type(node).__name__}")
    return float(eval(compile(tree, "<expr>", "eval"), {"__builtins__": {}}))


def calculate(expression: str) -> str:
    """
    A calculator that can evaluate arithmetic expressions. The input is a mathematical expression, as a string, and the output is the result of the calculation.

    Args:
        expression: the arithmetic expression to evaluate.

    Returns:
        The result of the calculation, as a string. Or error if the expression is invalid.
    """
    try:
        return str(evaluate_expression(expression))
    except Exception as e:
        return f"Error: {e}"


print(calculate("3 + 5"))
print(calculate("3 + "))

8.0
Error: invalid syntax (<unknown>, line 1)


### Docstring → JSON

The model never sees Python. The harness reads the signature and the docstring and turns them into JSON.

In [ ]:
def tool_schema(fn):
    sig = inspect.signature(fn)
    doc = inspect.getdoc(fn)

    print(sig)


    description, _, rest = doc.partition("Args:")
    description = " ".join(description.split())

    args_block = rest.partition("Returns:")[0]
    arg_descriptions = {}
    for line in args_block.strip().splitlines():
        name, _, text = line.strip().partition(":")
        if name and text:
            arg_descriptions[name.strip()] = text.strip()

    type_names = {str: "string", int: "integer", float: "number", bool: "boolean"}
    properties = {
        name: {"type": type_names.get(p.annotation, "string"),
               "description": arg_descriptions.get(name, "")}
        for name, p in sig.parameters.items()
    }
    return {"type": "function",
            "function": {"name": fn.__name__,
                         "description": description,
                         "parameters": {"type": "object",
                                        "properties": properties,
                                        "required": list(sig.parameters)}}}


schema = tool_schema(calculate)
print(json.dumps(schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "calculate",
    "description": "A calculator that can evaluate arithmetic expressions. The input is a mathematical expression, as a string, and the output is the result of the calculation.",
    "parameters": {
      "type": "object",
      "properties": {
        "expression": {
          "type": "string",
          "description": "the arithmetic expression to evaluate."
        }
      },
      "required": [
        "expression"
      ]
    }
  }
}


## 2. JSON → tokens (gpt-4o's tokenizer)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

schema_text = json.dumps(schema)
schema_tokens = [enc.decode([t]) for t in enc.encode(schema_text)]
print(len(schema_tokens), "tokens")
print(schema_tokens)

88 tokens
['{"', 'type', '":', ' "', 'function', '",', ' "', 'function', '":', ' {"', 'name', '":', ' "', 'calculate', '",', ' "', 'description', '":', ' "', 'A', ' calculator', ' that', ' can', ' evaluate', ' arithmetic', ' expressions', '.', ' The', ' input', ' is', ' a', ' mathematical', ' expression', ',', ' as', ' a', ' string', ',', ' and', ' the', ' output', ' is', ' the', ' result', ' of', ' the', ' calculation', '.",', ' "', 'parameters', '":', ' {"', 'type', '":', ' "', 'object', '",', ' "', 'properties', '":', ' {"', 'expression', '":', ' {"', 'type', '":', ' "', 'string', '",', ' "', 'description', '":', ' "', 'the', ' arithmetic', ' expression', ' to', ' evaluate', '."', '}},', ' "', 'required', '":', ' ["', 'expression', '"]', '}}', '}']


In [ ]:
call = {"name": "calculate", "arguments": {"expression": "3 + 5"}}
call_tokens = [enc.decode([t]) for t in enc.encode(json.dumps(call))]
print(len(call_tokens), "tokens")
print(call_tokens)

18 tokens
['{"', 'name', '":', ' "', 'calculate', '",', ' "', 'arguments', '":', ' {"', 'expression', '":', ' "', '3', ' +', ' ', '5', '"}}']


## 3. Where the JSON goes: the full prompt (Qwen2.5)

Qwen2.5's chat template pastes the JSON into the system prompt between `<tools>` tags.

In [ ]:
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

user = ("Calculate the following expression 3 + 5. Give your answer in the format "
        "<ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.")
messages = [{"role": "user", "content": user}]

prompt = tok.apply_chat_template(messages, tools=[schema], tokenize=False, add_generation_prompt=True)
print(prompt)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "calculate", "description": "A calculator that can evaluate arithmetic expressions. The input is a mathematical expression, as a string, and the output is the result of the calculation.", "parameters": {"type": "object", "properties": {"expression": {"type": "string", "description": "the arithmetic expression to evaluate."}}, "required": ["expression"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Calculate the following expression 3 + 5. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is

In [ ]:
prompt_tokens = [tok.decode([i]) for i in tok.encode(prompt, add_special_tokens=False)]
print(len(prompt_tokens), "tokens")
print(prompt_tokens)

230 tokens
['<|im_start|>', 'system', '\n', 'You', ' are', ' Q', 'wen', ',', ' created', ' by', ' Alibaba', ' Cloud', '.', ' You', ' are', ' a', ' helpful', ' assistant', '.\n\n', '#', ' Tools', '\n\n', 'You', ' may', ' call', ' one', ' or', ' more', ' functions', ' to', ' assist', ' with', ' the', ' user', ' query', '.\n\n', 'You', ' are', ' provided', ' with', ' function', ' signatures', ' within', ' <', 'tools', '></', 'tools', '>', ' XML', ' tags', ':\n', '<', 'tools', '>\n', '{"', 'type', '":', ' "', 'function', '",', ' "', 'function', '":', ' {"', 'name', '":', ' "', 'calculate', '",', ' "', 'description', '":', ' "', 'A', ' calculator', ' that', ' can', ' evaluate', ' arithmetic', ' expressions', '.', ' The', ' input', ' is', ' a', ' mathematical', ' expression', ',', ' as', ' a', ' string', ',', ' and', ' the', ' output', ' is', ' the', ' result', ' of', ' the', ' calculation', '.",', ' "', 'parameters', '":', ' {"', 'type', '":', ' "', 'object', '",', ' "', 'properties', '":',

In [ ]:
# the tags are single tokens in Qwen2.5's vocabulary
for s in ["<|im_start|>", "<|im_end|>", "<tool_call>", "</tool_call>"]:
    print(s, tok.encode(s, add_special_tokens=False))

<|im_start|> [151644]
<|im_end|> [151645]
<tool_call> [151657]
</tool_call> [151658]


## 4. The model only outputs text

**Before running:** the model "decides to use calculate". What does it do?
- A. runs the Python function itself and replies with 8.0
- B. writes a text message naming the tool and its arguments
- C. asks the user to run calculate for it
- D. edits the tool description to include the answer

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [ ]:
def generate(messages, tools, max_new_tokens=100):
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)


def show_messages(messages, max_chars=300):
    for m in messages:
        content = m["content"] if len(m["content"]) <= max_chars else m["content"][:max_chars] + " ..."
        print(f"{m['role']}: {content}\n")


reply = generate(messages, [schema])
show_messages(messages + [{"role": "assistant", "content": reply}])

user: Calculate the following expression 3 + 5. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.

assistant: <tool_call>
{"name": "calculate", "arguments": {"expression": "3 + 5"}}
</tool_call>



Answer: **B**. `calculate` has not been called.

The name and the argument key come from the JSON. Change the JSON and the output follows:

In [ ]:
import copy

schema_renamed = copy.deepcopy(schema)
schema_renamed["function"]["parameters"]["properties"] = {"expr": schema["function"]["parameters"]["properties"]["expression"]}
schema_renamed["function"]["parameters"]["required"] = ["expr"]
print("argument renamed to 'expr'")
print("assistant:", generate(messages, [schema_renamed]), "\n")

schema_renamed["function"]["name"] = "do_maths"
print("tool renamed to 'do_maths'")
print("assistant:", generate(messages, [schema_renamed]), "\n")

print("no tools")
print("assistant:", generate(messages, None))

argument renamed to 'expr'
assistant: <tool_call>
{"name": "calculate", "arguments": {"expr": "3 + 5"}}
</tool_call> 

tool renamed to 'do_maths'
assistant: <tool_call>
{"name": "do_maths", "arguments": {"expr": "3 + 5"}}
</tool_call> 

no tools
assistant: <ANSWER>NUMBER</ANSWER>1.0<float>


## 5. The harness

### `parse_tool_calls`: the reply string → a list of dicts

In [ ]:
def parse_tool_calls(text):
    calls = []
    for block in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL):
        calls.append(json.loads(block))
    return calls


print("before parsing:")
print(reply)
print()
tool_calls = parse_tool_calls(reply)
print("after parsing:")
print(tool_calls)

before parsing:
<tool_call>
{"name": "calculate", "arguments": {"expression": "3 + 5"}}
</tool_call>

after parsing:
[{'name': 'calculate', 'arguments': {'expression': '3 + 5'}}]


### `execute_tools`: look the function up by name, call it with the arguments

In [ ]:
tools = {"calculate": calculate}

def execute_tools(tool_calls, tools):
    tool_messages = []
    for call in tool_calls:
        fn = tools[call["name"]]

        result = fn(**call["arguments"])
        tool_messages.append({"role": "tool", "content": result})
    return tool_messages


tool_messages = execute_tools(tool_calls, tools)
print(tool_messages)

[{'role': 'tool', 'content': '8.0'}]


### Append, then generate again

The tool result is just more prompt text (Qwen wraps it in `<tool_response>` tags).

In [ ]:
messages.append({"role": "assistant", "content": reply})
messages.extend(tool_messages)

prompt2 = tok.apply_chat_template(messages, tools=[schema], tokenize=False, add_generation_prompt=True)
print(prompt2[len(prompt):])

<tool_call>
{"name": "calculate", "arguments": {"expression": "3 + 5"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
8.0
</tool_response><|im_end|>
<|im_start|>assistant



In [ ]:
messages.append({"role": "assistant", "content": generate(messages, [schema])})
show_messages(messages)

user: Calculate the following expression 3 + 5. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.

assistant: <tool_call>
{"name": "calculate", "arguments": {"expression": "3 + 5"}}
</tool_call>

tool: 8.0

assistant: The result of the expression 3 + 5 is <ANSWER>8.0</ANSWER>.



### The loop

In [ ]:
class ArithmeticTask:
    def __init__(self, problems):
        self.problems = problems                 # list of (expression, correct answer)
        self.current = 0

    def get_current_instruction(self):
        expression = self.problems[self.current][0]
        return {"role": "user", "content":
                f"Calculate the following expression {expression}. Give your answer in the format "
                f"<ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float."}

    def check_answer(self, answer):
        return answer is not None and abs(answer - self.problems[self.current][1]) < 1e-6

    def update_current_task(self):
        self.current += 1

    def all_answered(self):
        return self.current >= len(self.problems)


def extract_answer(text):
    m = re.search(r"<ANSWER>(.*?)</ANSWER>", text, flags=re.DOTALL)
    try:
        return float(m.group(1)) if m else None
    except ValueError:
        return None

In [ ]:
def arithmetic_agent(task, tools, message_limit=40):
    messages = []
    success = False
    while not success:
        messages.append(task.get_current_instruction())

        reply = generate(messages, [schema])
        messages.append({"role": "assistant", "content": reply})

        tool_calls = parse_tool_calls(reply)
        if tool_calls:
            messages.extend(execute_tools(tool_calls, tools))
            reply = generate(messages, [schema])
            messages.append({"role": "assistant", "content": reply})

        answer = extract_answer(reply)
        if task.check_answer(answer):
            task.update_current_task()
        else:
            messages.append({"role": "user", "content": "Incorrect answer. Try again."})

        if task.all_answered():
            success = True
        if len(messages) >= message_limit:
            print(f"message_limit={message_limit} reached, stopping")
            break
    return messages


task = ArithmeticTask([("3 + 5", 8.0), ("3 - 5", -2.0), ("(2 + 3) * 4", 20.0)])
show_messages(arithmetic_agent(task, tools, message_limit=40))

user: Calculate the following expression 3 + 5. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.

assistant: <tool_call>
{"name": "calculate", "arguments": {"expression": "3 + 5"}}
</tool_call>

tool: 8.0

assistant: The result of the expression 3 + 5 is <ANSWER>8.0</ANSWER>.

user: Calculate the following expression 3 - 5. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.

assistant: <tool_call>
{"name": "calculate", "arguments": {"expression": "3 - 5"}}
</tool_call>

tool: -2.0

assistant: The result of the expression 3 - 5 is <ANSWER>-2.0</ANSWER>.

user: Calculate the following expression (2 + 3) * 4. Give your answer in the format <ANSWER>NUMBER</ANSWER> where NUMBER is a numerical value formatted as a float.

assistant: <tool_call>
{"name": "calculate", "arguments": {"expression": "(2 + 3) * 4"}}
</tool_call>

tool: 20.0

assistant: The result of the expressi

### Why `message_limit` is not optional

With `Qwen2.5-1.5B-Instruct` the same loop calls the tool correctly, then answers in prose with no `<ANSWER>` tags, so `extract_answer` returns `None`, the loop says "Incorrect answer", and it repeats: 44 messages, 1 of 3 solved, stopped only by `message_limit=40`. Change `MODEL` and re-run from section 3 to see it.

### Malformed JSON

In [ ]:
bad_reply = '<tool_call>\n{"name": "calculate", "arguments": {"expression": "3 + 5"}\n</tool_call>'   # missing a brace
try:
    parse_tool_calls(bad_reply)
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

JSONDecodeError: Expecting ',' delimiter: line 1 column 59 (char 58)


## 6. A tool that calls Wikipedia

Same kind of Python function, but its body makes a web request. That request happens inside `execute_tools`.

In [ ]:
import wikipedia
from datetime import timedelta

wikipedia.set_user_agent("ARENA-AI-safety-course/3.4 (TARA)")              # Wikipedia rejects the default user-agent
wikipedia.set_rate_limiting(True, min_wait=timedelta(milliseconds=300))    # back-to-back requests sometimes come back empty

MAX_CHARS = 2000

def get_content(title: str) -> str:
    """
    Get the text content of a Wikipedia page.

    Args:
        title: the exact title of the Wikipedia page.

    Returns:
        The first part of the page's text, as a string. Or error if the page does not exist.
    """
    try:
        return wikipedia.page(title, auto_suggest=False).content[:MAX_CHARS]
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


wiki_schema = tool_schema(get_content)
print(json.dumps(wiki_schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_content",
    "description": "Get the text content of a Wikipedia page.",
    "parameters": {
      "type": "object",
      "properties": {
        "title": {
          "type": "string",
          "description": "the exact title of the Wikipedia page."
        }
      },
      "required": [
        "title"
      ]
    }
  }
}


In [ ]:
tools2 = {"calculate": calculate, "get_content": get_content}
schemas2 = [schema, wiki_schema]

wiki_messages = [{"role": "user", "content":
                  "What is the first sentence of the Wikipedia page 'Large language model'? "
                  "Give your answer in the format <ANSWER>SENTENCE</ANSWER>."}]

wiki_reply = generate(wiki_messages, schemas2)
wiki_messages.append({"role": "assistant", "content": wiki_reply})

wiki_calls = parse_tool_calls(wiki_reply)
print("after parsing:")
print(wiki_calls)
print()

wiki_messages.extend(execute_tools(wiki_calls, tools2))          # <- the Wikipedia request happens here
wiki_messages.append({"role": "assistant", "content": generate(wiki_messages, schemas2)})
show_messages(wiki_messages)

after parsing:
[{'name': 'get_content', 'arguments': {'title': 'Large language model'}}, {'name': 'calculate', 'arguments': {'expression': "len(get_content(title='Large language model'))-1"}}, {'name': 'get_content', 'arguments': {'title': 'Large language model'}}]

user: What is the first sentence of the Wikipedia page 'Large language model'? Give your answer in the format <ANSWER>SENTENCE</ANSWER>.

assistant: <tool_call>
{"name": "get_content", "arguments": {"title": "Large language model"}}
</tool_call>
<tool_call>
{"name": "calculate", "arguments": {"expression": "len(get_content(title='Large language model'))-1"}}
</tool_call>
<tool_call>
{"name": "get_content", "arguments": {"title": "Large language  ...

tool: A large language model (LLM) is a machine learning model (typically a neural network) trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contex

## 8. The Wikipedia API (what the WikiGame tools call)

The `wikipedia` package wraps Wikipedia's REST API. One call, one page object; `.content`, `.summary` and `.links` are the three things `GetContentTool`, `get_page_summary` and `get_permitted_links` return.

In [ ]:
%pip -q install wikipedia

In [ ]:
import wikipedia

page = wikipedia.page("Large language model", auto_suggest=False, redirect=True)

print(page.title)
print(page.url)
print(len(page.content), "characters of page text")
print(len(page.links), "links out of the page")
print()
print(page.summary[:300], "...")
print()
print(page.links[:15])

Large language model
https://en.wikipedia.org/wiki/Large_language_model
55366 characters of page text
833 links out of the page

A large language model (LLM) is a machine learning model (typically a neural network) trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts. They are the basis for m ...

['1.58-bit large language model', '15.ai', 'AAAI Conference on Artificial Intelligence', 'ACT-R', 'AI-assisted software development', 'AI-complete', 'AI-induced psychosis', 'AI21 Labs', 'AI agent', 'AI agent benchmark', 'AI alignment', 'AI anthropomorphism', 'AI boom', 'AI bubble', 'AI data center']


Two things to notice before you write the tools in 3.4:

* `page.links` is every link on the page, including ones that are not in the body text. ARENA's `get_permitted_links` keeps only those whose title also appears in `page.content`.
* `wikipedia.page` raises on ambiguous titles (`DisambiguationError`) and on misses (`PageError`). `MovePageTool` has to catch these, or one bad guess from the model ends the whole run.

## 9. The same steps, with the names you will use in 3.4

| in this notebook | in the 3.4 notebook (Inspect) | who does it |
|---|---|---|
| `calculate(expression)` + `tool_schema` | `@tool def calculate(): async def execute(...)` — Inspect reads the docstring | you write the docstring; Inspect makes the JSON |
| `tok.apply_chat_template(..., tools=...)` | inside `get_model().generate(input=state.messages, tools=[calculate()])` | the provider's template / API |
| `generate(messages, tools)` | `state.output = await get_model().generate(...)` | the model (text out) |
| `parse_tool_calls(reply)` | inside `generate` — `state.output.message.tool_calls` is already a list | Inspect (open model) or OpenAI's servers (API model) |
| `messages.append({"role": "assistant", ...})` | `state.messages.append(state.output.message)` — **manual** this week | you |
| `execute_tools(tool_calls, tools)` | `messages, state.output = await execute_tools(state.messages, tools=[calculate()])` | the harness, on your machine |
| `messages.extend(tool_messages)` | `state.messages.extend(messages)` | you |
| `while not success` | `@agent` — the loop you write | you |
| `message_limit=40` | `Task(..., message_limit=40)` | Inspect stops the run |
| `ArithmeticTask`, `extract_answer` | same names, in the notebook | you |
| `get_content(title)` | `GetContentTool` in the WikiGame section | you write it; it calls `wikipedia` |